# AeroSync-Mamba v33 — learnable instance-independent expert weighting: gradient-global (Part A) and class-aware (Part B)

**What this notebook is.** It replaces v32's abandoned per-window router with two weighting schemes that do **not**
depend on the individual window:
- **Part A — gradient-learned global weights.** Three scalars, `softmax(z_A, z_I, z_F1)`. This is v32's grid-search
  TUNED, learned by gradient descent instead. It is a sanity check and should land on TUNED.
- **Part B — class-aware weights.** A 3 × 5 matrix, with one expert-weight triple **per output class**. This is the new
  idea, and it is a small, deliberate step up in capacity (3 → 15 parameters).

**Why.** In v32, TUNED (one fixed weight triple fit by grid search) beat both I alone and the 995-parameter per-window
router. This dataset supports a *small* number of learned combination parameters, but not window-by-window adaptation.
Part B asks whether a *slightly* richer, still window-independent scheme earns its extra parameters.

**No expert is trained, re-trained or run in this notebook.** The inputs are:
- **`version 32/v32_npz/`:** the saved per-window test probabilities of v32's experts A, I and F1.
- **`version 32/ran-minor-project-v32.ipynb`:** v32's executed notebook. It is read **only** for its printed
  reference values (TUNED weights, temperatures, split sizes).
- **`version 29/Results/` and `version 30/Results/`:** the saved run CSVs, for the parameter-count table only.

`version 29/Npz/` is **never read**. v29's I probabilities differ from v32's for the same window ids, so they are not a
valid substitute. The loader refuses that path, and a hook confirms it was not touched.

> ### ⚠️ Known data gap: 28 of the 30 expected probability files exist
> `I_0_42.npz` and `I_1_123.npz` are missing from `version 32/v32_npz/`. The notebook does **not** hard-code this. It
> discovers from disk which `(expert, fold, seed)` files exist, and derives everything else from that:
> 1. **Evaluated pairs.** A `(fold, seed)` pair is evaluated only if **all three** experts have a file. With the current
>    files that is 8 of 10 pairs, and fold 0 / seed 42 and fold 1 / seed 123 cannot be evaluated.
> 2. **Reduced fitting pools.** v32 built each fold's CAL_HALF / ROUTER_HALF from **the other four days' saved test
>    windows**. A seed that is missing one day's I file therefore has a **3-day pool, not 4**, for *every* fold at that
>    seed, not only for the fold that cannot be evaluated. With the current files, **all 8 evaluated pairs have a
>    3-day pool.** Each pair's excluded day, and the reason, is printed. Nothing is padded or substituted.
> 3. **Pooled numbers.** Every pooled number is **over the evaluated pairs only (8 of 10)**. The windows of fold 0 are
>    predicted by seed 123 alone, and those of fold 1 by seed 42 alone. The pooled numbers are **not** comparable to
>    v32's 10-pair numbers, so every baseline is recomputed here on the same reduced set.

**How to run it on Kaggle so the outputs persist.** Use **Save Version → Save & Run All (Commit)**. Only a committed
Version keeps `/kaggle/working` (`v33_results_table.csv`, `v33_paired.csv`, `v33_summary.txt`). An interactive session's
outputs are lost when the session closes, which is what happened to version 29.

## 1. Setup and file discovery

In [1]:
# ==============================================================================
# 1a. SETUP — locate the project root (local checkout, Kaggle input, or a fresh clone of the public repo)
# ==============================================================================
import glob, hashlib, itertools, json, math, os, random, re, subprocess, sys, time, warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar

T0_NOTEBOOK = time.time()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "v33_outputs"
WORK.mkdir(parents=True, exist_ok=True)
NPZ_REL = "version 32/v32_npz"                                # the ONLY source of per-window probabilities
V32_RAN_REL = "version 32/ran-minor-project-v32.ipynb"        # read only for its printed reference values
RESULTS_RELS = ["version 29/Results", "version 30/Results"]    # parameter counts only
FORBIDDEN_NPZ_REL = "version 29/Npz"                          # never read: NOT a valid substitute for v32's files


def _has_inputs(root):
    return (root / NPZ_REL).is_dir() and (root / V32_RAN_REL).is_file()


_candidates = [Path.cwd().parent, Path.cwd()]
if Path("/kaggle/input").is_dir():
    _candidates += sorted(Path("/kaggle/input").glob("*")) + sorted(Path("/kaggle/input").glob("*/*"))
ROOT = next((r for r in _candidates if _has_inputs(r)), None)
if ROOT is None:
    ROOT = WORK / "AeroSync-Mamba"
    if not (ROOT / ".git").exists():
        print("Cloning the public project repo (--depth 1) to read version 32's saved files ...")
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Sainava/AeroSync-Mamba.git", str(ROOT)], check=True)
print(f"Project root: {ROOT}")
NPZ_DIR = (ROOT / NPZ_REL).resolve()
V32_RAN = ROOT / V32_RAN_REL
V29_RUNS_CSV = ROOT / "version 29" / "Results" / "gated_fusion_results.csv"
V30_RUNS_CSV = ROOT / "version 30" / "Results" / "crossattn_results.csv"

# ---- constants (identical to v32 where shared) ----
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
SMALL_FOLD_DAY = "2021-12-22"
SEEDS = [42, 123]
EXPERTS = ["A", "I", "F1"]
N_CLASSES = 5
EXPECTED_N_WINDOWS, EXPECTED_N_MISSIONS = 5238, 95
SPLIT_SEED = 7
T_GRID = np.unique(np.concatenate([np.exp(np.linspace(np.log(0.05), np.log(20.0), 400)), [1.0]]))
TUNED_GRID_STEP = 0.05
BOOTSTRAP_RESAMPLES, BOOTSTRAP_SEED = 2000, 2028
NOISE_FLOOR_F1 = 0.016
# ---- new in v33: gradient-learned weights (fixed a priori; nothing is selected on a test day) ----
WEIGHT_EPOCHS = 3000          # full-batch Adam steps on ROUTER_HALF, no early stopping
WEIGHT_LR = 0.05
# ---- Part A sanity-check thresholds (fixed a priori) ----
L1_CLOSE, L1_LARGE = 0.10, 0.20   # grid step 0.05 => the grid optimum can sit up to ~0.1 (L1) from the continuous one
NLL_TOL = 1e-3                    # the continuous optimum must be at least as good as the best 0.05-grid point, up to this
PROB_TOL = 1e-9
HOOKS = defaultdict(list)


def hook(name, ok, msg, stop=False):
    ok = bool(ok)
    HOOKS[name].append((ok, msg))
    print(f"[HOOK {name}] {'PASS' if ok else 'FAIL'} — {msg}")
    if not ok and stop:
        raise RuntimeError(f"HOOK {name} FAILED: {msg}")
    return ok


def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


LOADED_NPZ = []                   # every .npz path this notebook reads, for the provenance hook


def load_npz(path):
    """The only way this notebook reads a .npz: records the path and refuses anything outside version 32/v32_npz."""
    path = Path(path).resolve()
    if FORBIDDEN_NPZ_REL in path.as_posix() or path.parent != NPZ_DIR:
        raise RuntimeError(f"refusing to read {path}: only {NPZ_DIR} is a permitted probability source")
    LOADED_NPZ.append(path)
    return np.load(path)


for rel in [NPZ_REL] + RESULTS_RELS:
    d = ROOT / rel
    names = sorted(f.name for f in d.iterdir()) if d.is_dir() else None
    print(f"\n{rel}/ " + ("<< FOLDER NOT FOUND >>" if names is None else f"({len(names)} files)"))
    for n in names or []:
        print(f"    {n}")
hook("inputs", NPZ_DIR.is_dir() and V32_RAN.is_file(), f"{NPZ_REL}/ and {V32_RAN_REL} found under {ROOT}", stop=True)

Cloning the public project repo (--depth 1) to read version 32's saved files ...


Cloning into '/kaggle/working/AeroSync-Mamba'...


Project root: /kaggle/working/AeroSync-Mamba

version 32/v32_npz/ (28 files)
    A_0_123.npz
    A_0_42.npz
    A_1_123.npz
    A_1_42.npz
    A_2_123.npz
    A_2_42.npz
    A_3_123.npz
    A_3_42.npz
    A_4_123.npz
    A_4_42.npz
    F1_0_123.npz
    F1_0_42.npz
    F1_1_123.npz
    F1_1_42.npz
    F1_2_123.npz
    F1_2_42.npz
    F1_3_123.npz
    F1_3_42.npz
    F1_4_123.npz
    F1_4_42.npz
    I_0_123.npz
    I_1_42.npz
    I_2_123.npz
    I_2_42.npz
    I_3_123.npz
    I_3_42.npz
    I_4_123.npz
    I_4_42.npz

version 29/Results/ (3 files)
    gated_fusion_results.csv
    v29_results_table.csv
    v29_summary.txt

version 30/Results/ (3 files)
    crossattn_results.csv
    v30_results_table.csv
    v30_summary.txt
[HOOK inputs] PASS — version 32/v32_npz/ and version 32/ran-minor-project-v32.ipynb found under /kaggle/working/AeroSync-Mamba


True

In [2]:
# ==============================================================================
# 1b. FILE DISCOVERY — which (expert, fold, seed) files exist; derive the evaluable pairs from that (nothing hard-coded)
# ==============================================================================
_NPZ_RE = re.compile(r"^(A|I|F1)_(\d)_(\d+)\.npz$")
FOUND = {}
_names = sorted(f.name for f in NPZ_DIR.glob("*.npz"))
for n in _names:
    m_ = _NPZ_RE.match(n)
    if m_:
        FOUND[(m_.group(1), int(m_.group(2)), int(m_.group(3)))] = NPZ_DIR / n
IGNORED = [n for n in _names if not _NPZ_RE.match(n)]
EXPECTED_FILES = [(m, k, s) for m in EXPERTS for k in range(len(DAYS)) for s in SEEDS]
MISSING = [key for key in EXPECTED_FILES if key not in FOUND]
UNEXPECTED = sorted(set(FOUND) - set(EXPECTED_FILES))

print(f"Expected {len(EXPECTED_FILES)} files ({EXPERTS} x folds 0-{len(DAYS) - 1} x seeds {SEEDS}) in {NPZ_DIR}")
for m in EXPERTS:
    f_ = [f"{m}_{k}_{s}.npz" for k in range(len(DAYS)) for s in SEEDS if (m, k, s) in FOUND]
    x_ = [f"{m}_{k}_{s}.npz" for k in range(len(DAYS)) for s in SEEDS if (m, k, s) not in FOUND]
    print(f"    {m:<3} found {len(f_):>2}/10: {', '.join(f_)}")
    print(f"        MISSING {len(x_)}: {', '.join(x_) if x_ else '(none)'}")
print(f"TOTAL: found {len(EXPECTED_FILES) - len(MISSING)}/{len(EXPECTED_FILES)}; MISSING: "
      + (", ".join(f"{m}_{k}_{s}.npz" for m, k, s in MISSING) if MISSING else "(none)"))
if IGNORED or UNEXPECTED:
    print(f"Ignored (name does not match, or outside folds/seeds): {IGNORED + [p.name for k_, p in FOUND.items() if k_ in UNEXPECTED]}")
    for k_ in UNEXPECTED:
        FOUND.pop(k_)

ALL_PAIRS = [(k, s) for k in range(len(DAYS)) for s in SEEDS]
EVAL_PAIRS = [(k, s) for k, s in ALL_PAIRS if all((m, k, s) in FOUND for m in EXPERTS)]
EXCLUDED_PAIRS = [(k, s) for k, s in ALL_PAIRS if (k, s) not in EVAL_PAIRS]
SEEDS_EVAL = [s for s in SEEDS if any(s2 == s for _, s2 in EVAL_PAIRS)]
FOLDS_BY_SEED = {s: [k for k, s2 in EVAL_PAIRS if s2 == s] for s in SEEDS_EVAL}
SEEDS_BY_FOLD = {k: [s for k2, s in EVAL_PAIRS if k2 == k] for k in range(len(DAYS))}
FOLDS_EVAL = [k for k in range(len(DAYS)) if SEEDS_BY_FOLD[k]]
# days whose windows can enter a seed's fitting pool: all three experts have that (day, seed) file
POOL_DAYS_OK = {s: [k for k in range(len(DAYS)) if all((m, k, s) in FOUND for m in EXPERTS)] for s in SEEDS}

print(f"\nEVALUATED (fold, seed) pairs — all three experts have a file: {len(EVAL_PAIRS)} of {len(ALL_PAIRS)}")
for k, s in EVAL_PAIRS:
    print(f"    fold {k} ({DAYS[k]}) seed {s}")
print(f"EXCLUDED pairs — at least one expert has no predictions for that fold's own test windows: {len(EXCLUDED_PAIRS)}")
for k, s in EXCLUDED_PAIRS:
    print(f"    fold {k} ({DAYS[k]}) seed {s}: missing " + ", ".join(f"{m}_{k}_{s}.npz" for m in EXPERTS if (m, k, s) not in FOUND)
          + " -> excluded from EVERY fit, evaluation and pooled statistic below")
print("\nPer seed: evaluated folds " + "; ".join(f"seed {s}: {FOLDS_BY_SEED[s]}" for s in SEEDS_EVAL))
print("Per fold: seeds available " + "; ".join(f"fold {k}: {SEEDS_BY_FOLD[k]}" for k in range(len(DAYS))))
N_EVAL = len(EVAL_PAIRS)
DENOM = (f"pooled over {N_EVAL} of {len(ALL_PAIRS)} fold/seed pairs (NOT {len(ALL_PAIRS)}); "
         + "; ".join(f"fold {k} windows by seed{'s' if len(SEEDS_BY_FOLD[k]) > 1 else ''} {SEEDS_BY_FOLD[k]}"
                     for k in FOLDS_EVAL))
print("\nDENOMINATOR used for every pooled number in this notebook: " + DENOM)
hook("discovery", N_EVAL > 0 and all(len(FOLDS_BY_SEED[s]) >= 2 for s in SEEDS_EVAL),
     f"{N_EVAL} evaluable pairs; every evaluated seed has >= 2 folds (needed for paired statistics)", stop=True)

Expected 30 files (['A', 'I', 'F1'] x folds 0-4 x seeds [42, 123]) in /kaggle/working/AeroSync-Mamba/version 32/v32_npz
    A   found 10/10: A_0_42.npz, A_0_123.npz, A_1_42.npz, A_1_123.npz, A_2_42.npz, A_2_123.npz, A_3_42.npz, A_3_123.npz, A_4_42.npz, A_4_123.npz
        MISSING 0: (none)
    I   found  8/10: I_0_123.npz, I_1_42.npz, I_2_42.npz, I_2_123.npz, I_3_42.npz, I_3_123.npz, I_4_42.npz, I_4_123.npz
        MISSING 2: I_0_42.npz, I_1_123.npz
    F1  found 10/10: F1_0_42.npz, F1_0_123.npz, F1_1_42.npz, F1_1_123.npz, F1_2_42.npz, F1_2_123.npz, F1_3_42.npz, F1_3_123.npz, F1_4_42.npz, F1_4_123.npz
        MISSING 0: (none)
TOTAL: found 28/30; MISSING: I_0_42.npz, I_1_123.npz

EVALUATED (fold, seed) pairs — all three experts have a file: 8 of 10
    fold 0 (2021-12-22) seed 123
    fold 1 (2021-12-31) seed 42
    fold 2 (2022-01-04) seed 42
    fold 2 (2022-01-04) seed 123
    fold 3 (2022-01-05) seed 42
    fold 3 (2022-01-05) seed 123
    fold 4 (2022-01-10) seed 42
    fold 4 (20

True

In [3]:
# ==============================================================================
# 1c. SCHEMA — read from an actual file, not assumed
# ==============================================================================
_first = sorted(FOUND)[0]
_z = load_npz(FOUND[_first])
SCHEMA = {k: (tuple(_z[k].shape), str(_z[k].dtype)) for k in _z.files}
print(f"Schema of {FOUND[_first].name}:")
for k, (sh, dt) in SCHEMA.items():
    print(f"    {k:<10} shape {sh}  dtype {dt}")
EXPECTED_KEYS = {"window_id", "mission", "day", "cls", "probs"}
HAS_LOGITS = any("logit" in k.lower() for k in SCHEMA)
print(f"Expected keys present: {EXPECTED_KEYS <= set(SCHEMA)} | extra keys: {sorted(set(SCHEMA) - EXPECTED_KEYS)} | "
      f"logits present: {HAS_LOGITS} -> calibration = probability-based power-law temperature p^(1/T), as in v32")
hook("schema", EXPECTED_KEYS <= set(SCHEMA) and SCHEMA["probs"][0][1:] == (N_CLASSES,),
     f"keys {sorted(SCHEMA)}; probs is (N, {N_CLASSES})", stop=True)

Schema of A_0_42.npz:
    window_id  shape (602,)  dtype <U32
    mission    shape (602,)  dtype <U27
    day        shape (602,)  dtype <U10
    cls        shape (602,)  dtype int64
    phase      shape (602,)  dtype <U6
    probs      shape (602, 5)  dtype float32
Expected keys present: True | extra keys: ['phase'] | logits present: False -> calibration = probability-based power-law temperature p^(1/T), as in v32
[HOOK schema] PASS — keys ['cls', 'day', 'mission', 'phase', 'probs', 'window_id']; probs is (N, 5)


True

In [4]:
# ==============================================================================
# 1d. LOAD + ALIGN — missing (expert, fold, seed) blocks stay NaN (never padded, never substituted)
# ==============================================================================
RAW, TEST_ONLY = {}, True
for (m, k, s), path in sorted(FOUND.items()):
    z = load_npz(path)
    d = {key: z[key] for key in ("window_id", "mission", "day", "cls", "probs")}
    if (d["day"] != DAYS[k]).any():
        TEST_ONLY = False
    RAW[(m, k, s)] = d
INNER_VAL_AVAILABLE = not TEST_ONLY
print(f"Files contain ONLY their own fold's test-day windows: {TEST_ONLY}")
if INNER_VAL_AVAILABLE:
    raise NotImplementedError("inner-validation windows were detected; this notebook implements only v32's fallback pools")

# window metadata for EVERY day comes from whichever files cover it (A and F1 cover all days); all files for a day must agree
_meta_parts = []
for k in range(len(DAYS)):
    keys = [key for key in sorted(RAW, key=lambda t: (EXPERTS.index(t[0]), t[2])) if key[1] == k]
    if not keys:
        loud(f"No file of any expert covers fold {k} ({DAYS[k]}); its windows are unknown.")
        continue
    ref = RAW[keys[0]]
    o = np.argsort(ref["window_id"])
    for key in keys[1:]:
        o2 = np.argsort(RAW[key]["window_id"])
        for f_ in ("window_id", "mission", "cls"):
            assert np.array_equal(ref[f_][o], RAW[key][f_][o2]), f"{key} disagrees with {keys[0]} on {f_}"
    _meta_parts.append(pd.DataFrame({f_: ref[f_][o] for f_ in ("window_id", "mission", "day", "cls")}))
META = pd.concat(_meta_parts).sort_values("window_id").reset_index(drop=True)
META["fold"] = META.day.map({d: i for i, d in enumerate(DAYS)})
ORDER_IDS = META.window_id.values
Y = META.cls.values.astype(int)
N = len(META)
ROW_OF = {w: i for i, w in enumerate(ORDER_IDS)}

P = {}          # P[(expert, seed)] -> (N, 5) raw probabilities aligned to ORDER_IDS; rows of a missing file stay NaN
AVAIL = {}      # AVAIL[(expert, seed)] -> boolean (N,) rows that come from a real file
for m in EXPERTS:
    for s in SEEDS:
        arr, av = np.full((N, N_CLASSES), np.nan), np.zeros(N, bool)
        for k in range(len(DAYS)):
            if (m, k, s) not in RAW:
                continue
            d = RAW[(m, k, s)]
            rows = np.array([ROW_OF[w] for w in d["window_id"]])
            pr = np.clip(d["probs"].astype(np.float64), 1e-12, None)
            arr[rows] = pr / pr.sum(1, keepdims=True)
            av[rows] = True
        P[(m, s)], AVAIL[(m, s)] = arr, av
IDS_SHA = hashlib.sha1("\n".join(ORDER_IDS).encode()).hexdigest()[:16]
hook("windows", N == EXPECTED_N_WINDOWS and META.mission.nunique() == EXPECTED_N_MISSIONS,
     f"window metadata covers {N} windows / {META.mission.nunique()} missions (window-id sha1 {IDS_SHA}); every file "
     "covering a day agrees on its window ids, missions and classes", stop=True)
for m in EXPERTS:
    for s in SEEDS:
        want = np.isin(META.fold.values, [k for k in range(len(DAYS)) if (m, k, s) in FOUND])
        assert np.array_equal(AVAIL[(m, s)], want) and np.isfinite(P[(m, s)][want]).all() and np.isnan(P[(m, s)][~want]).all()
print("Rows present per (expert, seed): " + "; ".join(f"{m}/{s}: {int(AVAIL[(m, s)].sum())}" for m in EXPERTS for s in SEEDS))

Files contain ONLY their own fold's test-day windows: True
[HOOK windows] PASS — window metadata covers 5238 windows / 95 missions (window-id sha1 1d23df9ff89e449b); every file covering a day agrees on its window ids, missions and classes
Rows present per (expert, seed): A/42: 5238; A/123: 5238; I/42: 4636; I/123: 3264; F1/42: 5238; F1/123: 5238


In [5]:
# ==============================================================================
# 1e. VERSION 32's REFERENCE VALUES — parsed from the executed v32 notebook's saved outputs, cross-checked to a transcription
# ==============================================================================
# transcription of ran-minor-project-v32.ipynb's printed output (used only if that file cannot be parsed)
V32_TUNED_TRANSCRIBED = {(0, 42): (0.0, 0.6, 0.4), (0, 123): (0.0, 0.85, 0.15), (1, 42): (0.0, 0.5, 0.5),
                         (1, 123): (0.0, 0.75, 0.25), (2, 42): (0.0, 0.6, 0.4), (2, 123): (0.0, 0.85, 0.15),
                         (3, 42): (0.0, 0.75, 0.25), (3, 123): (0.0, 0.95, 0.05), (4, 42): (0.0, 0.45, 0.55),
                         (4, 123): (0.0, 0.75, 0.25)}
V32_T_TRANSCRIBED = {(0, 42): (3.475, 2.297, 3.314), (0, 123): (3.963, 1.322, 4.476), (1, 42): (3.734, 2.008, 3.838),
                     (1, 123): (4.267, 1.579, 4.515), (2, 42): (2.171, 2.463, 1.975), (2, 123): (2.122, 1.359, 2.498),
                     (3, 42): (4.394, 2.869, 4.022), (3, 123): (4.768, 1.677, 5.285), (4, 42): (3.525, 2.175, 3.13),
                     (4, 123): (3.491, 1.297, 3.729)}
V32_SPLITS_TRANSCRIBED = {0: (42, 2245, 43, 2391), 1: (29, 1603, 30, 1661), 2: (38, 2109, 40, 2192),
                          3: (39, 2168, 41, 2280), 4: (38, 2053, 40, 2250)}

_txt = []
for c in json.load(open(V32_RAN, encoding="utf-8"))["cells"]:
    for o in c.get("outputs", []):
        if "text" in o:
            _txt.append("".join(o["text"]))
        elif "text/plain" in o.get("data", {}):
            _txt.append("".join(o["data"]["text/plain"]))
_txt = re.sub(r"np\.float64\(([^()]*)\)", r"\1", "\n".join(_txt))
V32_TUNED, V32_T, V32_SPLITS = {}, {}, {}
for mm in re.finditer(r"fold (\d) seed (\d+): TUNED weights \(A, I, F1\) = \(([^)]*)\).*?T \(A, I, F1\) = \(([^)]*)\)", _txt):
    key = (int(mm.group(1)), int(mm.group(2)))
    V32_TUNED[key] = tuple(float(x) for x in mm.group(3).split(","))
    V32_T[key] = tuple(float(x) for x in mm.group(4).split(","))
for mm in re.finditer(r"fold (\d) \([\d-]+(?:, SMALL)?\): CAL_HALF (\d+) missions / (\d+) windows, "
                      r"ROUTER_HALF (\d+) missions / (\d+) windows", _txt):
    V32_SPLITS[int(mm.group(1))] = tuple(int(mm.group(i)) for i in range(2, 6))
_parsed_ok = len(V32_TUNED) == len(ALL_PAIRS) and len(V32_SPLITS) == len(DAYS)
hook("v32_reference", _parsed_ok and V32_TUNED == V32_TUNED_TRANSCRIBED and V32_T == V32_T_TRANSCRIBED
     and V32_SPLITS == V32_SPLITS_TRANSCRIBED,
     f"parsed {len(V32_TUNED)} TUNED weight triples, {len(V32_T)} temperature triples and {len(V32_SPLITS)} split sizes from "
     f"{V32_RAN.name}'s saved outputs; identical to the transcription")
if not _parsed_ok:
    loud("Could not parse v32's executed outputs — using the TRANSCRIBED copy (a reported claim, not re-read from the file).")
    V32_TUNED, V32_T, V32_SPLITS = V32_TUNED_TRANSCRIBED, V32_T_TRANSCRIBED, V32_SPLITS_TRANSCRIBED
print("v32 TUNED (A, I, F1) per fold/seed: " + "; ".join(f"{k}/{s} {V32_TUNED[(k, s)]}" for k, s in ALL_PAIRS))

[HOOK v32_reference] PASS — parsed 10 TUNED weight triples, 10 temperature triples and 5 split sizes from ran-minor-project-v32.ipynb's saved outputs; identical to the transcription
v32 TUNED (A, I, F1) per fold/seed: 0/42 (0.0, 0.6, 0.4); 0/123 (0.0, 0.85, 0.15); 1/42 (0.0, 0.5, 0.5); 1/123 (0.0, 0.75, 0.25); 2/42 (0.0, 0.6, 0.4); 2/123 (0.0, 0.85, 0.15); 3/42 (0.0, 0.75, 0.25); 3/123 (0.0, 0.95, 0.05); 4/42 (0.0, 0.45, 0.55); 4/123 (0.0, 0.75, 0.25)


In [6]:
# ==============================================================================
# v29 helper functions, VERBATIM from version 29/minor-project-v29.ipynb (cell 18): compute_metrics, mission_level
# ==============================================================================
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score)
from scipy import stats
LABELS = list(range(N_CLASSES))


def compute_metrics(y, P):
    pred = P.argmax(1)
    try:
        auroc = float(roc_auc_score(y, P, multi_class="ovr", average="macro", labels=LABELS))
    except ValueError:
        auroc = float("nan")
    return {"accuracy": float(accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=LABELS, zero_division=0)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)), "macro_auroc": auroc}


def mission_level(missions, y, P):
    df = pd.DataFrame(P, columns=LABELS)
    df["mission"], df["y"] = missions, y
    g = df.groupby("mission", sort=True)
    return g["y"].first().values, g[LABELS].mean().values, g["y"].first().index.values

In [7]:
# ==============================================================================
# v29 helper functions, VERBATIM from version 29/minor-project-v29.ipynb (cell 24): prf_from_cm, cluster_bootstrap_f1, level_block
# ==============================================================================



def prf_from_cm(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return prec, rec, f1


def cluster_bootstrap_f1(missions, y, pred, B=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    """Macro-F1 over B resamples of MISSIONS within class (never windows)."""
    rng = np.random.default_rng(seed)
    ms = pd.Series(missions).unique()
    pos = {m: i for i, m in enumerate(ms)}
    cnt, mcls = np.zeros((len(ms), N_CLASSES), int), np.zeros(len(ms), int)
    for m, t, p in zip(missions, y, pred):
        cnt[pos[m], p] += 1
        mcls[pos[m]] = t
    cms = np.zeros((B, N_CLASSES, N_CLASSES))
    for c in range(N_CLASSES):
        rows = np.where(mcls == c)[0]
        cms[:, c, :] = cnt[rows[rng.integers(0, len(rows), (B, len(rows)))]].sum(1)
    return np.array([prf_from_cm(cm)[2].mean() for cm in cms])


def level_block(missions, y, P):
    pred = P.argmax(1)
    m = compute_metrics(y, P)
    cm = confusion_matrix(y, pred, labels=LABELS)
    _, rec, _ = prf_from_cm(cm)
    lo, hi = np.percentile(cluster_bootstrap_f1(missions, y, pred), [2.5, 97.5])
    err = pred != y
    return {"accuracy": m["accuracy"], "macro_f1": m["macro_f1"], "f1_ci_low": lo, "f1_ci_high": hi,
            "balanced_accuracy": m["balanced_accuracy"], "macro_auroc": m["macro_auroc"],
            **{f"recall_c{c}": rec[c] for c in range(N_CLASSES)},
            "faulty_pred_normal": float((pred[y > 0] == 0).mean()), "normal_pred_faulty": float((pred[y == 0] > 0).mean()),
            "adjacent_error_share": float((np.abs(pred - y)[err] == 1).mean()) if err.any() else float("nan"), "cm": cm}

## 2. CAL_HALF / ROUTER_HALF — v32's split, then restricted to the days that actually have files

**Step 1. The split is v32's, verbatim.** For outer fold k, the pool is the other four days' missions. These are split
by mission, stratified by class, with `default_rng([7, k])`. The split depends only on mission / day / class
metadata, which is complete for all 95 missions because A and F1 cover every day. So it reproduces v32's halves
exactly, and its sizes are checked against v32's printed values.

**Step 2. Restriction per (fold, seed).** A day's windows can enter seed s's CAL_HALF / ROUTER_HALF only if **all three**
experts have that day's seed-s file. Otherwise the day is **excluded from that seed's pool**, and the notebook prints
which day and which file. The remaining missions keep exactly the half they had in v32. The pool shrinks (3 days
instead of 4), but it is not re-drawn, which keeps it as comparable to v32 as the data allows.

Day k itself is never in fold k's pool, as in v32. Caveats carried over from v32: the pool windows carry the *other*
folds' expert checkpoints, which is weak indirect leakage; this is not a nested CV.

In [8]:
# ==============================================================================
# 2. SPLITS — v32's exact split, then the per-(fold, seed) reduced pool
# ==============================================================================
SPLITS = {}
for k, day in enumerate(DAYS):
    pool = META[META.day != day]
    cal, rou = [], []
    rng = np.random.default_rng([SPLIT_SEED, k])
    for c in range(N_CLASSES):
        ms = sorted(pool[pool.cls == c].mission.unique())
        perm = rng.permutation(len(ms))
        h = len(ms) // 2
        cal += [ms[i] for i in perm[:h]]
        rou += [ms[i] for i in perm[h:]]
    test_idx = np.where(META.day.values == day)[0]
    cal_idx = np.where(META.mission.isin(cal).values)[0]
    rou_idx = np.where(META.mission.isin(rou).values)[0]
    SPLITS[k] = {"cal": cal_idx, "router": rou_idx, "test": test_idx, "cal_m": set(cal), "rou_m": set(rou)}
    got = (len(cal), len(cal_idx), len(rou), len(rou_idx))
    hook("v32_split", got == V32_SPLITS[k],
         f"fold {k} ({day}): full v32 split reproduced — CAL_HALF {got[0]} missions / {got[1]} windows, ROUTER_HALF "
         f"{got[2]} missions / {got[3]} windows (v32 printed {V32_SPLITS[k]})")

POOLS, POOL_ROWS = {}, []
print()
for k, s in EVAL_PAIRS:
    other = [j for j in range(len(DAYS)) if j != k]
    used = [j for j in other if j in POOL_DAYS_OK[s]]
    dropped = [j for j in other if j not in POOL_DAYS_OK[s]]
    keep = np.isin(META.fold.values, used)
    ci, ri, ti = SPLITS[k]["cal"][keep[SPLITS[k]["cal"]]], SPLITS[k]["router"][keep[SPLITS[k]["router"]]], SPLITS[k]["test"]
    POOLS[(k, s)] = {"cal": ci, "router": ri, "test": ti, "days_used": used, "days_dropped": dropped}
    why = "; ".join(f"day {DAYS[j]} (fold {j}) EXCLUDED — missing "
                    + ", ".join(f"{m}_{j}_{s}.npz" for m in EXPERTS if (m, j, s) not in FOUND) for j in dropped)
    print(f"fold {k} seed {s}: pool = {len(used)} of {len(other)} other days {[DAYS[j] for j in used]}"
          + (f"  |  {why}" if dropped else "  |  full 4-day pool")
          + f"  |  CAL_HALF {len(set(META.mission.values[ci]))} missions / {len(ci)} windows "
            f"(v32 full: {len(SPLITS[k]['cal'])}), ROUTER_HALF {len(set(META.mission.values[ri]))} missions / {len(ri)} windows "
            f"(v32 full: {len(SPLITS[k]['router'])}), test {len(ti)} windows")
    POOL_ROWS.append({"fold": k, "seed": s, "pool_days": len(used), "days_used": ",".join(DAYS[j] for j in used),
                      "days_excluded": ",".join(DAYS[j] for j in dropped) or "-", "cal_windows": len(ci),
                      "router_windows": len(ri), "test_windows": len(ti)})
    all_finite = all(np.isfinite(P[(m, s)][np.concatenate([ci, ri, ti])]).all() for m in EXPERTS)
    disjoint = not (set(ci) & set(ri)) and not ((set(ci) | set(ri)) & set(ti)) and DAYS[k] not in set(META.day.values[np.concatenate([ci, ri])])
    hook("pools", all_finite and disjoint,
         f"fold {k} seed {s}: CAL/ROUTER/test disjoint, day {DAYS[k]} not in the pool, every row has real predictions from all "
         "three experts (no NaN = no missing file used)")
    for name, idx in (("CAL_HALF", ci), ("ROUTER_HALF", ri)):
        present = set(Y[idx])
        if present != set(range(N_CLASSES)):
            loud(f"fold {k} seed {s}: {name} lacks class(es) {sorted(set(range(N_CLASSES)) - present)} after the pool reduction")
POOL_TAB = pd.DataFrame(POOL_ROWS)
print("\nPool composition per evaluated pair:")
print(POOL_TAB.to_string(index=False))
print(f"\n{int((POOL_TAB.pool_days < len(DAYS) - 1).sum())} of {len(POOL_TAB)} evaluated pairs use a REDUCED pool "
      f"(< {len(DAYS) - 1} other days).")
hook("excluded_pairs", not (set(POOLS) & set(EXCLUDED_PAIRS)),
     f"excluded pairs {EXCLUDED_PAIRS} have no pool, no calibration, no fit and no evaluation")

[HOOK v32_split] PASS — fold 0 (2021-12-22): full v32 split reproduced — CAL_HALF 42 missions / 2245 windows, ROUTER_HALF 43 missions / 2391 windows (v32 printed (42, 2245, 43, 2391))
[HOOK v32_split] PASS — fold 1 (2021-12-31): full v32 split reproduced — CAL_HALF 29 missions / 1603 windows, ROUTER_HALF 30 missions / 1661 windows (v32 printed (29, 1603, 30, 1661))
[HOOK v32_split] PASS — fold 2 (2022-01-04): full v32 split reproduced — CAL_HALF 38 missions / 2109 windows, ROUTER_HALF 40 missions / 2192 windows (v32 printed (38, 2109, 40, 2192))
[HOOK v32_split] PASS — fold 3 (2022-01-05): full v32 split reproduced — CAL_HALF 39 missions / 2168 windows, ROUTER_HALF 41 missions / 2280 windows (v32 printed (39, 2168, 41, 2280))
[HOOK v32_split] PASS — fold 4 (2022-01-10): full v32 split reproduced — CAL_HALF 38 missions / 2053 windows, ROUTER_HALF 40 missions / 2250 windows (v32 printed (38, 2053, 40, 2250))

fold 0 seed 123: pool = 3 of 4 other days ['2022-01-04', '2022-01-05', '2022-01

True

## 3. Calibration — v32's probability-based power-law temperature, re-fit on each pair's (reduced) CAL_HALF

$$p_{cal} = \frac{p^{1/T}}{\sum p^{1/T}}$$

This is v32's code, verbatim. T minimizes NLL on CAL_HALF over a 401-point log grid on [0.05, 20] that includes T = 1,
followed by a bounded refinement. Because CAL_HALF is now reduced, the temperatures differ from v32's. v32's values are
printed alongside for context only.

In [9]:
# ==============================================================================
# 3. CALIBRATION (functions verbatim from v32; fit per evaluated (fold, seed) on its reduced CAL_HALF)
# ==============================================================================
def calibrate(p, T):
    lp = np.log(p) / T
    lp -= lp.max(1, keepdims=True)
    q = np.exp(lp)
    return q / q.sum(1, keepdims=True)


def nll(p, y):
    return float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, None))))


def fit_T(p, y):
    grid = [nll(calibrate(p, T), y) for T in T_GRID]
    i = int(np.argmin(grid))
    lo, hi = np.log(T_GRID[max(i - 1, 0)]), np.log(T_GRID[min(i + 1, len(T_GRID) - 1)])
    r = minimize_scalar(lambda lt: nll(calibrate(p, math.exp(lt)), y), bounds=(lo, hi), method="bounded")
    return (math.exp(r.x), r.fun) if r.fun < grid[i] else (float(T_GRID[i]), grid[i])


CAL_T, CAL_ROWS = {}, []
for k, s in EVAL_PAIRS:
    ci = POOLS[(k, s)]["cal"]
    for i, m in enumerate(EXPERTS):
        before = nll(P[(m, s)][ci], Y[ci])
        T, after = fit_T(P[(m, s)][ci], Y[ci])
        CAL_T[(m, k, s)] = T
        CAL_ROWS.append({"fold": k, "seed": s, "expert": m, "T": T, "T v32 (4-day pool)": V32_T[(k, s)][i],
                         "NLL before": before, "NLL after": after, "reduced": after <= before + 1e-12})
CAL_TAB = pd.DataFrame(CAL_ROWS)
print("Fitted temperatures on the REDUCED CAL_HALF (v32's 4-day-pool values alongside, context only) and NLL before -> after:")
print(CAL_TAB.pivot_table(index=["fold", "seed"], columns="expert",
                          values=["T", "T v32 (4-day pool)", "NLL before", "NLL after"]).round(4).to_string())
_bad = CAL_TAB[~CAL_TAB.reduced]
hook("calibration", _bad.empty, "calibration did not increase CAL_HALF NLL for any expert/fold/seed (T = 1 is a candidate)"
     + ("" if _bad.empty else f" — exceptions: {_bad.to_dict('records')}"))
# Q[(m, k, s)]: expert m's seed-s probabilities calibrated with fold k's temperature; rows without a real file stay NaN
Q = {}
for k, s in EVAL_PAIRS:
    for m in EXPERTS:
        q = np.full((N, N_CLASSES), np.nan)
        av = AVAIL[(m, s)]
        q[av] = calibrate(P[(m, s)][av], CAL_T[(m, k, s)])
        Q[(m, k, s)] = q

Fitted temperatures on the REDUCED CAL_HALF (v32's 4-day-pool values alongside, context only) and NLL before -> after:
          NLL after                 NLL before                       T                 T v32 (4-day pool)              
expert            A      F1       I          A      F1       I       A      F1       I                  A     F1      I
fold seed                                                                                                              
0    123     0.4875  0.4173  0.2784     1.4361  1.3802  0.3048  5.0310  5.0733  1.4446              3.963  4.476  1.322
1    42      0.5024  0.3803  0.3836     1.0689  0.9530  0.4461  3.7110  4.1385  1.7525              3.734  3.838  2.008
2    42      0.2871  0.1665  0.3804     0.3492  0.2154  0.5188  1.8679  1.9139  2.2700              2.171  1.975  2.463
     123     0.3581  0.2207  0.2711     0.5893  0.3696  0.3070  2.7964  2.7124  1.5622              2.122  2.498  1.359
3    42      0.6053  0.4520  0.3751     1

## 4. Methods — TUNED (v32's grid), Part A (gradient global), Part B (class-aware)

Notation: $q_e(c)$ is expert $e \in \{A, I, F1\}$'s calibrated probability of class $c$ for a window.

| method | learnable parameters | window-dependent? | combination |
|---|---|---|---|
| **EQUAL** | 0 | no | $p(c) = \tfrac13 \sum_e q_e(c)$ |
| **TUNED** (v32, re-fit here) | 3 weights on the simplex, 0.05 grid (231 points) | no | $p(c) = \sum_e w_e\, q_e(c)$, with $w$ = argmin NLL on ROUTER_HALF |
| **PART A** (new) | 3 logits $z_e$ (2 effective) | no | $w = \mathrm{softmax}(z)$, $\;p(c) = \sum_e w_e\, q_e(c)$ |
| **PART B** (new) | 3 × 5 logits $Z_{e,c}$ (10 effective) | no, class-dependent | see below |

**The Part B combination rule.** Each class has its own weight triple, obtained with a softmax over the experts
separately for each class:

$$W_{e,c} = \frac{\exp Z_{e,c}}{\sum_{e'} \exp Z_{e',c}}, \qquad \tilde p(c) = \sum_e W_{e,c}\, q_e(c), \qquad
p(c) = \frac{\tilde p(c)}{\sum_{c'} \tilde p(c')}$$

*Why this is a valid distribution.*
- $\tilde p(c)$ is a convex combination of the three experts' mass on class $c$. So
  $\min_e q_e(c) \le \tilde p(c) \le \max_e q_e(c)$, which makes it strictly positive, because the calibrated
  probabilities are strictly positive.
- The class columns use different weights, so $\sum_c \tilde p(c)$ is generally **not** 1. It lies in
  $(0, 3]$. Dividing by it gives a vector that is non-negative and sums to 1.
- The renormalization multiplies every class by the same positive number for a given window, so **it never changes
  the argmax**. It matters only for the cross-entropy, AUROC and mission-level averaging.

*Why this rule.*
- **Nesting.** When all five columns of $W$ are equal to one triple $w$, $\sum_c \tilde p(c) = 1$ and Part B is
  exactly Part A. So Part B strictly contains Part A, and on ROUTER_HALF its fitted NLL can only be ≤ Part A's (a
  hook checks this).
- **Starting point.** Both start from $Z = 0$, which is EQUAL.
- **The check.** Validity is checked numerically on every test window of every evaluated pair (sum = 1, no negative
  values, finite).

**Training (Parts A and B identical).**
- Cross-entropy $-\log p(y)$ on the pair's ROUTER_HALF, full batch.
- Adam, lr 0.05, **3000 steps fixed in advance**, no early stopping, no weight decay, float64.
- Initialization $z = 0$ / $Z = 0$.

**Seeding.** The optimization has **no random component**: zero initialization, full batch, no dropout, no shuffling.
The result is therefore deterministic. `random`, NumPy and torch are still seeded with **the same seed s as the expert
probabilities being combined** (42 or 123), and `torch.use_deterministic_algorithms(True)` is set. There is no third
notion of seed.

The per-window router from v32 is **not** rebuilt. For contrast only: it was an MLP `27 → 32 → 3` with 995
parameters, and it produced window-dependent weights from confidence and disagreement features.

In [10]:
# ==============================================================================
# 4. METHODS — TUNED grid (v32 verbatim), PART A and PART B (gradient, torch CPU, float64)
# ==============================================================================
import torch

torch.use_deterministic_algorithms(True)
DT = torch.float64


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def softmax_np(z, axis=0):
    e = np.exp(z - z.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def mix_global_t(z, E):
    """E: (n, 3, 5) calibrated expert probabilities; z: (3,) logits. Returns (n, 5)."""
    w = torch.softmax(z, dim=0)
    return (w[None, :, None] * E).sum(1)


def mix_class_t(Z, E):
    """Z: (3, 5) logits, one expert-weight triple per class (softmax over experts, per class). Returns (n, 5), rows sum to 1."""
    W = torch.softmax(Z, dim=0)
    pt = (W[None, :, :] * E).sum(1)
    return pt / pt.sum(1, keepdim=True)


def apply_global(z, qs):
    w = softmax_np(np.asarray(z, dtype=np.float64), axis=0)
    return sum(w[i] * qs[m] for i, m in enumerate(EXPERTS))


def apply_class(Z, qs):
    W = softmax_np(np.asarray(Z, dtype=np.float64), axis=0)                 # (3, 5), columns sum to 1
    pt = sum(W[i][None, :] * qs[m] for i, m in enumerate(EXPERTS))           # (n, 5)
    return pt / pt.sum(1, keepdims=True)


def fit_weights(kind, qs, y, seed):
    """Full-batch Adam on ROUTER_HALF cross-entropy. kind 'global' -> z (3,), 'class' -> Z (3, 5). Zero init (= EQUAL)."""
    seed_all(seed)
    shape = (len(EXPERTS),) if kind == "global" else (len(EXPERTS), N_CLASSES)
    mix = mix_global_t if kind == "global" else mix_class_t
    z = torch.zeros(shape, dtype=DT, requires_grad=True)
    opt = torch.optim.Adam([z], lr=WEIGHT_LR)
    E = torch.tensor(np.stack([qs[m] for m in EXPERTS], axis=1), dtype=DT)
    yt = torch.tensor(y, dtype=torch.long)
    ar = torch.arange(len(yt))

    def loss_fn():
        return -torch.log(mix(z, E)[ar, yt].clamp_min(1e-300)).mean()

    first = None
    for _ in range(WEIGHT_EPOCHS):
        opt.zero_grad()
        loss = loss_fn()
        if first is None:
            first = loss.item()
        loss.backward()
        opt.step()
    opt.zero_grad()
    loss = loss_fn()
    loss.backward()
    return z.detach().numpy().copy(), first, loss.item(), float(z.grad.norm())


SIMPLEX = [np.array(w) for w in itertools.product(np.round(np.arange(0, 1 + 1e-9, TUNED_GRID_STEP), 4), repeat=3)
           if abs(sum(w) - 1) < 1e-9]
print(f"TUNED grid: {len(SIMPLEX)} weight triples (step {TUNED_GRID_STEP}); Part A: {len(EXPERTS)} logits; "
      f"Part B: {len(EXPERTS)} x {N_CLASSES} = {len(EXPERTS) * N_CLASSES} logits; {WEIGHT_EPOCHS} Adam steps, lr {WEIGHT_LR}")

TUNED grid: 231 weight triples (step 0.05); Part A: 3 logits; Part B: 3 x 5 = 15 logits; 3000 Adam steps, lr 0.05


In [11]:
# ==============================================================================
# 4b. FIT EVERY METHOD ON EACH EVALUATED PAIR; PREDICT ITS OWN TEST DAY
# ==============================================================================
PART_A, PART_B = "PART A (global)", "PART B (class-aware)"
METHODS = ["A (cal)", "I (cal)", "F1 (cal)", "EQUAL", "TUNED", PART_A, PART_B, "ORACLE"]
OUT = {m: {s: np.full((N, N_CLASSES), np.nan) for s in SEEDS} for m in METHODS}   # test probabilities; NaN = not evaluated
COVER = {s: np.zeros(N, bool) for s in SEEDS}
FIT = {}
_t = time.time()
for k, s in EVAL_PAIRS:
    r_idx, t_idx = POOLS[(k, s)]["router"], POOLS[(k, s)]["test"]
    qs = {m: Q[(m, k, s)] for m in EXPERTS}
    qr = {m: qs[m][r_idx] for m in EXPERTS}
    qt = {m: qs[m][t_idx] for m in EXPERTS}
    assert all(np.isfinite(qr[m]).all() and np.isfinite(qt[m]).all() for m in EXPERTS)
    for m in EXPERTS:
        OUT[f"{m} (cal)"][s][t_idx] = qt[m]
    OUT["EQUAL"][s][t_idx] = np.mean([qt[m] for m in EXPERTS], axis=0)
    # TUNED: v32's grid search, verbatim, on this pair's ROUTER_HALF
    best = min(SIMPLEX, key=lambda w: nll(sum(w[i] * qr[m] for i, m in enumerate(EXPERTS)), Y[r_idx]))
    OUT["TUNED"][s][t_idx] = sum(best[i] * qt[m] for i, m in enumerate(EXPERTS))
    nll_tuned = nll(sum(best[i] * qr[m] for i, m in enumerate(EXPERTS)), Y[r_idx])
    # PART A and PART B
    zA, a0, a1, ga = fit_weights("global", qr, Y[r_idx], s)
    zB, b0, b1, gb = fit_weights("class", qr, Y[r_idx], s)
    OUT[PART_A][s][t_idx] = apply_global(zA, qt)
    pB = apply_class(zB, qt)
    OUT[PART_B][s][t_idx] = pB
    nll_A, nll_B = nll(apply_global(zA, qr), Y[r_idx]), nll(apply_class(zB, qr), Y[r_idx])
    # ORACLE (per window, v32): true class if any expert's argmax is right, else the EQUAL argmax (one-hot)
    preds = np.stack([qt[m].argmax(1) for m in EXPERTS])
    op = np.where((preds == Y[t_idx]).any(0), Y[t_idx], OUT["EQUAL"][s][t_idx].argmax(1))
    OUT["ORACLE"][s][t_idx] = np.eye(N_CLASSES)[op]
    COVER[s][t_idx] = True
    FIT[(k, s)] = {"tuned": best, "wA": softmax_np(zA), "WB": softmax_np(zB, axis=0), "nll_tuned": nll_tuned,
                   "nll_A": nll_A, "nll_B": nll_B, "loss_A": (a0, a1), "loss_B": (b0, b1), "grad_A": ga, "grad_B": gb}
    print(f"fold {k} seed {s}: TUNED {tuple(np.round(best, 2))} NLL {nll_tuned:.4f} | PART A w {tuple(np.round(FIT[(k, s)]['wA'], 3))} "
          f"CE {a0:.4f} -> {a1:.4f} (|grad| {ga:.1e}) | PART B CE {b0:.4f} -> {b1:.4f} (|grad| {gb:.1e}) | "
          f"T (A, I, F1) = {tuple(round(CAL_T[(m, k, s)], 3) for m in EXPERTS)}")
    # Part B validity: EVERY test window, plus a printed sample of 3
    ok = np.isfinite(pB).all() and (pB >= 0).all() and np.abs(pB.sum(1) - 1).max() < PROB_TOL
    hook("partB_valid", ok, f"fold {k} seed {s}: all {len(t_idx)} test windows — finite, min {pB.min():.2e} >= 0, "
                            f"max |sum - 1| {np.abs(pB.sum(1) - 1).max():.1e}")
    for j in np.linspace(0, len(t_idx) - 1, 3).astype(int):
        print(f"      sample window {ORDER_IDS[t_idx[j]]}: p = {np.round(pB[j], 4).tolist()}  sum {pB[j].sum():.12f}")
    hook("partB_nests_A", nll_B <= nll_A + NLL_TOL,
         f"fold {k} seed {s}: ROUTER_HALF NLL Part B {nll_B:.4f} <= Part A {nll_A:.4f} + {NLL_TOL} (B contains A as a special case)")
print(f"All fits: {time.time() - _t:.1f} s")
for s in SEEDS:
    for k in range(len(DAYS)):
        rows = META.fold.values == k
        if (k, s) in EXCLUDED_PAIRS:
            assert not COVER[s][rows].any() and all(np.isnan(OUT[m][s][rows]).all() for m in METHODS)
hook("excluded_pairs", all(not COVER[s][META.fold.values == k].any() for k, s in EXCLUDED_PAIRS),
     f"{EXCLUDED_PAIRS}: no test prediction exists for any method (all NaN) — they cannot enter any statistic below")

fold 0 seed 123: TUNED (np.float64(0.0), np.float64(0.75), np.float64(0.25)) NLL 0.4009 | PART A w (np.float64(0.0), np.float64(0.729), np.float64(0.271)) CE 0.4555 -> 0.4008 (|grad| 8.7e-06) | PART B CE 0.4555 -> 0.3729 (|grad| 5.3e-06) | T (A, I, F1) = (5.031, 1.445, 5.073)
[HOOK partB_valid] PASS — fold 0 seed 123: all 602 test windows — finite, min 1.31e-07 >= 0, max |sum - 1| 4.4e-16
      sample window rosbag2_2021_12_22-09_14_31#0000: p = [0.918, 0.0229, 0.0122, 0.0308, 0.0161]  sum 1.000000000000
      sample window rosbag2_2021_12_22-09_37_54#0001: p = [0.0039, 0.0047, 0.6589, 0.3326, 0.0]  sum 1.000000000000
      sample window rosbag2_2021_12_22-09_54_38#0057: p = [0.062, 0.2856, 0.176, 0.2592, 0.2173]  sum 1.000000000000
[HOOK partB_nests_A] PASS — fold 0 seed 123: ROUTER_HALF NLL Part B 0.3729 <= Part A 0.4008 + 0.001 (B contains A as a special case)
fold 1 seed 42: TUNED (np.float64(0.0), np.float64(0.55), np.float64(0.45)) NLL 0.3205 | PART A w (np.float64(0.0), np.float

True

## 5. Sanity check — does Part A reproduce TUNED?

Two comparisons are made for every evaluated pair:
1. **Part A vs TUNED re-fit here, on the same reduced pool and the same temperatures.** This is the implementation
   check. The gradient optimum over the continuous simplex must reach an NLL no worse than the best 0.05-grid point
   (tolerance 1e-3).
   - A higher NLL points to an optimization or implementation **bug**.
   - A large weight distance with an *equal* NLL means the NLL surface is **flat**, not that something is wrong.
2. **Part A vs v32's printed TUNED weights** (4-day pool, v32's temperatures). Every evaluated pair here has a
   **reduced** pool, so differences in this column mix two sources: the change in data and a possible bug.
   Comparison 1 separates them.

Distances are the L1 distance between weight vectors: close ≤ 0.10, LARGE > 0.20. Thresholds are fixed in advance.

In [12]:
# ==============================================================================
# 5. PART A vs TUNED
# ==============================================================================
SAN = []
for k, s in EVAL_PAIRS:
    f = FIT[(k, s)]
    v32w = np.array(V32_TUNED[(k, s)])
    l1_here, l1_v32, l1_tt = (np.abs(f["wA"] - f["tuned"]).sum(), np.abs(f["wA"] - v32w).sum(), np.abs(f["tuned"] - v32w).sum())
    nll_ok = f["nll_A"] <= f["nll_tuned"] + NLL_TOL
    status = ("BUG? Part A's NLL is worse than the grid optimum" if not nll_ok else
              "close" if l1_here <= L1_CLOSE else
              "moderate (NLL fine)" if l1_here <= L1_LARGE else "LARGE distance but NLL fine -> flat optimum, not a bug")
    SAN.append({"fold": k, "seed": s, "pool days": len(POOLS[(k, s)]["days_used"]),
                "v32 TUNED (A,I,F1)": tuple(np.round(v32w, 2)), "v33 TUNED refit": tuple(np.round(f["tuned"], 2)),
                "PART A (A,I,F1)": tuple(np.round(f["wA"], 3)), "L1 A-v33TUNED": l1_here, "L1 A-v32TUNED": l1_v32,
                "L1 v33-v32 TUNED": l1_tt, "NLL TUNED": f["nll_tuned"], "NLL A": f["nll_A"],
                "dNLL (A-TUNED)": f["nll_A"] - f["nll_tuned"], "implementation check": status,
                "vs v32": "close" if l1_v32 <= L1_CLOSE else "moderate" if l1_v32 <= L1_LARGE else "LARGE (data change or bug)"})
SAN = pd.DataFrame(SAN)
print(SAN.round(4).to_string(index=False))
for r in SAN.to_dict("records"):
    hook("partA_vs_tuned", not r["implementation check"].startswith("BUG"),
         f"fold {r['fold']} seed {r['seed']}: Part A {r['PART A (A,I,F1)']} vs v33 TUNED {r['v33 TUNED refit']} "
         f"L1 {r['L1 A-v33TUNED']:.3f}, dNLL {r['dNLL (A-TUNED)']:+.5f} -> {r['implementation check']} | vs v32 TUNED "
         f"{r['v32 TUNED (A,I,F1)']} L1 {r['L1 A-v32TUNED']:.3f} ({r['vs v32']})")
_large = SAN[(SAN["L1 A-v33TUNED"] > L1_LARGE) | (SAN["L1 A-v32TUNED"] > L1_LARGE)]
if len(_large):
    loud("LARGE weight discrepancy for: " + ", ".join(f"fold {r.fold} seed {r.seed}" for r in _large.itertuples())
         + "\nRead the 'implementation check' column: an NLL worse than TUNED's = bug; equal NLL = flat optimum; a large "
           "distance only from v32 (not from v33's refit) = the reduced pool / refit temperatures, not this code.")
SANITY_STATEMENT = (
    f"Part A vs TUNED re-fit on the same reduced pool: median L1 {SAN['L1 A-v33TUNED'].median():.3f} "
    f"(max {SAN['L1 A-v33TUNED'].max():.3f}); Part A's ROUTER_HALF NLL minus TUNED's: max {SAN['dNLL (A-TUNED)'].max():+.5f} "
    f"({'no' if SAN['implementation check'].str.startswith('BUG').sum() == 0 else SAN['implementation check'].str.startswith('BUG').sum()} "
    f"implementation failures). Part A vs v32's printed TUNED (4-day pool): median L1 {SAN['L1 A-v32TUNED'].median():.3f} "
    f"(max {SAN['L1 A-v32TUNED'].max():.3f}); v33's own TUNED refit vs v32's: median L1 {SAN['L1 v33-v32 TUNED'].median():.3f}.")
print("\n" + SANITY_STATEMENT)

 fold  seed  pool days v32 TUNED (A,I,F1)   v33 TUNED refit      PART A (A,I,F1)  L1 A-v33TUNED  L1 A-v32TUNED  L1 v33-v32 TUNED  NLL TUNED  NLL A  dNLL (A-TUNED) implementation check                     vs v32
    0   123          3  (0.0, 0.85, 0.15) (0.0, 0.75, 0.25)  (0.0, 0.729, 0.271)         0.0426         0.2426               0.2     0.4009 0.4008         -0.0001                close LARGE (data change or bug)
    1    42          3    (0.0, 0.5, 0.5) (0.0, 0.55, 0.45)  (0.0, 0.572, 0.428)         0.0443         0.1443               0.1     0.3205 0.3204         -0.0001                close                   moderate
    2    42          3    (0.0, 0.6, 0.4) (0.0, 0.65, 0.35)  (0.0, 0.642, 0.358)         0.0164         0.0837               0.1     0.3008 0.3008         -0.0000                close                      close
    2   123          3  (0.0, 0.85, 0.15) (0.05, 0.7, 0.25) (0.061, 0.719, 0.22)         0.0601         0.2627               0.3     0.2407 0.2406         -

## 6. Results — pooled over the evaluated fold/seed pairs only (8 of 10, **not** 10)

Each window's test probabilities are averaged over the seeds that were evaluated **for that window's fold**. With the
current files:
- folds 2–4 average seeds 42 and 123;
- fold 0 uses seed 123 only;
- fold 1 uses seed 42 only.

Every baseline is recomputed on this same set. **v32's 10-pair pooled numbers are not comparable and are not reused.**

The metrics are v29's own functions: accuracy, macro-F1 with a 95 % mission-level cluster bootstrap CI, balanced
accuracy, macro-AUROC, per-class recall and the safety breakdown. ORACLE's AUROC is meaningless and is reported as NaN.

In [13]:
# ==============================================================================
# 6. RESULTS TABLE (pooled over EVAL_PAIRS only)
# ==============================================================================
N_COVER = sum(COVER[s].astype(int) for s in SEEDS)
ROWS = N_COVER > 0
hook("coverage", ROWS.sum() == np.isin(META.fold.values, FOLDS_EVAL).sum(),
     f"{int(ROWS.sum())} of {N} windows are predicted by >= 1 evaluated seed ({int((N_COVER == 2).sum())} by 2 seeds, "
     f"{int((N_COVER == 1).sum())} by 1 seed); windows of folds with no evaluated seed are dropped: "
     f"{int((~ROWS).sum())}")


def pool_over_seeds(per_seed):
    acc = np.zeros((N, N_CLASSES))
    for s in SEEDS:
        acc[COVER[s]] += per_seed[s][COVER[s]]
    out = np.full((N, N_CLASSES), np.nan)
    out[ROWS] = acc[ROWS] / N_COVER[ROWS][:, None]
    return out


POOLED = {m: pool_over_seeds(OUT[m]) for m in METHODS if m != "ORACLE"}
_pe = np.stack([POOLED[f"{e} (cal)"].argmax(1) for e in EXPERTS])
_or = np.full((N, N_CLASSES), np.nan)
_or[ROWS] = np.eye(N_CLASSES)[np.where((_pe == Y).any(0), Y, POOLED["EQUAL"].argmax(1))][ROWS]
POOLED["ORACLE"] = _or
assert all(np.isfinite(POOLED[m][ROWS]).all() for m in METHODS)
MIS = META.mission.values[ROWS]
YR = Y[ROWS]


def oracle_mission(prob_by_expert):
    """Mission-level oracle from each expert's mission-level prediction (argmax of its mean window probability), ROWS only."""
    preds, ym, names = {}, None, None
    for m, p in prob_by_expert.items():
        ym, pm, names = mission_level(MIS, YR, p[ROWS])
        preds[m] = pm.argmax(1)
    any_ok = np.any([preds[m] == ym for m in preds], axis=0)
    eq = mission_level(MIS, YR, np.mean([p[ROWS] for p in prob_by_expert.values()], axis=0))[1].argmax(1)
    return ym, np.eye(N_CLASSES)[np.where(any_ok, ym, eq)], names


RESULTS = []
for m in METHODS:
    for level in ("window", "mission"):
        if level == "window":
            b = level_block(MIS, YR, POOLED[m][ROWS])
        elif m == "ORACLE":
            ym, pm, names = oracle_mission({e: POOLED[f"{e} (cal)"] for e in EXPERTS})
            b = level_block(names, ym, pm)
        else:
            ym, pm, names = mission_level(MIS, YR, POOLED[m][ROWS])
            b = level_block(names, ym, pm)
        if m == "ORACLE":
            b["macro_auroc"] = float("nan")
        RESULTS.append({"method": m, "level": level, "n_pairs_pooled": N_EVAL, "pooling": DENOM,
                        **{k_: v for k_, v in b.items() if k_ != "cm"}})
RESULTS = pd.DataFrame(RESULTS)
RESULTS.to_csv(WORK / "v33_results_table.csv", index=False)
pd.set_option("display.width", 250)
for level in ("window", "mission"):
    t = RESULTS[RESULTS.level == level].set_index("method")
    t["macro-F1 [95% CI]"] = [f"{a:.4f} [{b:.4f}, {c:.4f}]" for a, b, c in zip(t.macro_f1, t.f1_ci_low, t.f1_ci_high)]
    print(f"\n=== {level.upper()} LEVEL — {DENOM} ===")
    print(t[["accuracy", "macro-F1 [95% CI]", "balanced_accuracy", "macro_auroc"] + [f"recall_c{c}" for c in range(N_CLASSES)]
            + ["faulty_pred_normal", "normal_pred_faulty", "adjacent_error_share"]].round(4).to_string())
print("\nThese numbers are over the reduced evaluation set above and are NOT comparable to v32's 10-pair pooled table.")

[HOOK coverage] PASS — 5238 of 5238 windows are predicted by >= 1 evaluated seed (2662 by 2 seeds, 2576 by 1 seed); windows of folds with no evaluated seed are dropped: 0

=== WINDOW LEVEL — pooled over 8 of 10 fold/seed pairs (NOT 10); fold 0 windows by seed [123]; fold 1 windows by seed [42]; fold 2 windows by seeds [42, 123]; fold 3 windows by seeds [42, 123]; fold 4 windows by seeds [42, 123] ===
                      accuracy        macro-F1 [95% CI]  balanced_accuracy  macro_auroc  recall_c0  recall_c1  recall_c2  recall_c3  recall_c4  faulty_pred_normal  normal_pred_faulty  adjacent_error_share
method                                                                                                                                                                                                      
A (cal)                 0.7827  0.7864 [0.7289, 0.8394]             0.7853       0.9319      1.000     0.8327     0.6366     0.6753     0.7819              0.0005               0.000    

## 7. Paired comparisons

1. **Within each seed, over that seed's own evaluated folds.** With the current files this is seed 42 on folds 1–4
   and seed 123 on folds 0, 2, 3 and 4, so n = 4 each. The two seeds have different fold sets, so they are **not**
   paired against each other.
2. **Combined at the end, over every fold with at least one evaluated seed** (n = 5), with both of v32's methods:
   - **pooled-probability:** fold macro-F1 of the seed-pooled probabilities;
   - **seed-averaged-F1:** for each fold, the mean over *its available seeds* of that seed's fold macro-F1.

   Folds 0 and 1 therefore enter from one seed each. A disagreement between the two methods in wins or sign is flagged.

**Two verdict rules, both fixed in advance.**
- **v32 rule** (generalized to n folds): **BEATS** if mean Δ > 0.016 **and** wins ≥ ⌈0.8 n⌉. **WORSE** if mean Δ < −0.016
  **and** wins ≤ ⌊0.2 n⌋.
- **Noise rule:** a **clear** difference means |mean Δ| > the SD of Δ across the same folds.

**The primary test of the new idea is Part B vs Part A.** Part B "clearly beats" Part A only if mean Δ > SD under
**both** combined methods at window level.

In [14]:
# ==============================================================================
# 7. PAIRED COMPARISONS
# ==============================================================================
def oracle_mission_fold(sel, experts_probs):
    preds, ym, names = {}, None, None
    for e, p in experts_probs.items():
        ym, pm, names = mission_level(META.mission.values[sel], Y[sel], p[sel])
        preds[e] = pm.argmax(1)
    any_ok = np.any([preds[e] == ym for e in preds], axis=0)
    eq = mission_level(META.mission.values[sel], Y[sel], np.mean([p[sel] for p in experts_probs.values()], axis=0))[1].argmax(1)
    return ym, np.eye(N_CLASSES)[np.where(any_ok, ym, eq)], names


def fold_f1(method, level, k, source):
    """Macro-F1 of `method` on fold k. source: a seed (that seed's own predictions) or 'pooled'."""
    sel = META.fold.values == k
    get = (lambda name: POOLED[name]) if source == "pooled" else (lambda name: OUT[name][source])
    if method == "ORACLE" and level == "mission":
        ep = {e: get(f"{e} (cal)") for e in EXPERTS}
        assert all(np.isfinite(p[sel]).all() for p in ep.values())
        ym, pm, _ = oracle_mission_fold(sel, ep)
        return f1_score(ym, pm.argmax(1), average="macro", labels=LABELS, zero_division=0)
    probs = get(method)
    assert np.isfinite(probs[sel]).all(), f"{method} fold {k} source {source}: NaN rows — an excluded pair leaked in"
    if level == "window":
        return f1_score(Y[sel], probs[sel].argmax(1), average="macro", labels=LABELS, zero_division=0)
    ym, pm, _ = mission_level(META.mission.values[sel], Y[sel], probs[sel])
    return f1_score(ym, pm.argmax(1), average="macro", labels=LABELS, zero_division=0)


def paired(a, b):
    d = np.asarray(a) - np.asarray(b)
    n = len(d)
    sd = d.std(ddof=1) if n > 1 else float("nan")
    p = stats.ttest_rel(a, b).pvalue if n > 1 and sd > 0 else float("nan")
    return {"n": n, "mean": d.mean(), "sd": sd, "wins": int((d > 0).sum()), "p": p, "diffs": np.round(d, 4).tolist()}


def verdict(st):
    n = st["n"]
    return ("BEATS" if st["mean"] > NOISE_FLOOR_F1 and st["wins"] >= math.ceil(0.8 * n) else
            "WORSE" if st["mean"] < -NOISE_FLOOR_F1 and st["wins"] <= math.floor(0.2 * n) else "no reliable difference")


def noise_rule(st):
    if not np.isfinite(st["sd"]):
        return "n/a"
    if st["sd"] == 0:
        return "identical on every fold" if st["mean"] == 0 else ("clear gain" if st["mean"] > 0 else "clear loss")
    return "clear gain (mean > SD)" if st["mean"] > st["sd"] else "clear loss (-mean > SD)" if -st["mean"] > st["sd"] else "within noise"


def fmt(st):
    return (f"mean Δ {st['mean']:+.4f}  SD {st['sd']:.4f}  wins {st['wins']}/{st['n']}  p {st['p']:.3f}  per fold {st['diffs']}"
            f"  -> v32 rule: {verdict(st)} | noise rule: {noise_rule(st)}")


COMPARISONS = [(PART_A, "I (cal)"), (PART_A, "TUNED"), (PART_B, PART_A), (PART_B, "I (cal)"), (PART_B, "ORACLE")]
PAIRED, PAIRED_ROWS = [], []
for a_m, b_m in COMPARISONS:
    for level in ("window", "mission"):
        row = {"comparison": f"{a_m} − {b_m}", "level": level}
        print(f"\n{row['comparison']} [{level}]")
        for s in SEEDS_EVAL:
            ks = FOLDS_BY_SEED[s]
            st = paired([fold_f1(a_m, level, k, s) for k in ks], [fold_f1(b_m, level, k, s) for k in ks])
            row[f"seed {s}"] = st
            print(f"   seed {s:<4} folds {ks}  {fmt(st)}")
        pa = [fold_f1(a_m, level, k, "pooled") for k in FOLDS_EVAL]
        pb = [fold_f1(b_m, level, k, "pooled") for k in FOLDS_EVAL]
        sa = [np.mean([fold_f1(a_m, level, k, s) for s in SEEDS_BY_FOLD[k]]) for k in FOLDS_EVAL]
        sb = [np.mean([fold_f1(b_m, level, k, s) for s in SEEDS_BY_FOLD[k]]) for k in FOLDS_EVAL]
        row["pooled-probability"], row["seed-averaged-F1"] = paired(pa, pb), paired(sa, sb)
        for meth in ("pooled-probability", "seed-averaged-F1"):
            print(f"   COMBINED {meth:<19} folds {FOLDS_EVAL}  {fmt(row[meth])}")
        pp, sf = row["pooled-probability"], row["seed-averaged-F1"]
        row["flag"] = ("⚠️ METHOD DISAGREEMENT (wins or sign differ) — both shown" if pp["wins"] != sf["wins"] or
                       np.sign(pp["mean"]) != np.sign(sf["mean"]) else "methods agree")
        print(f"   {row['flag']}")
        PAIRED.append(row)
        for key in [f"seed {s}" for s in SEEDS_EVAL] + ["pooled-probability", "seed-averaged-F1"]:
            st = row[key]
            PAIRED_ROWS.append({"comparison": row["comparison"], "level": level, "analysis": key, "n_folds": st["n"],
                                "mean_delta": st["mean"], "sd": st["sd"], "wins": st["wins"], "p": st["p"],
                                "per_fold": st["diffs"], "v32_rule": verdict(st), "noise_rule": noise_rule(st)})
pd.DataFrame(PAIRED_ROWS).to_csv(WORK / "v33_paired.csv", index=False)
print(f"\nCombined analyses: folds 0 and 1 enter with one seed each ({DENOM}).")


PART A (global) − I (cal) [window]
   seed 42   folds [1, 2, 3, 4]  mean Δ +0.0348  SD 0.0278  wins 3/4  p 0.088  per fold [0.0354, 0.038, 0.0669, -0.001]  -> v32 rule: no reliable difference | noise rule: clear gain (mean > SD)
   seed 123  folds [0, 2, 3, 4]  mean Δ +0.0033  SD 0.0080  wins 3/4  p 0.468  per fold [0.0103, -0.0071, 0.0014, 0.0086]  -> v32 rule: no reliable difference | noise rule: within noise
   COMBINED pooled-probability  folds [0, 1, 2, 3, 4]  mean Δ +0.0170  SD 0.0172  wins 4/5  p 0.091  per fold [0.0103, 0.0354, 0.0254, 0.0231, -0.0093]  -> v32 rule: BEATS | noise rule: within noise
   COMBINED seed-averaged-F1    folds [0, 1, 2, 3, 4]  mean Δ +0.0198  SD 0.0143  wins 5/5  p 0.036  per fold [0.0103, 0.0354, 0.0154, 0.0341, 0.0038]  -> v32 rule: BEATS | noise rule: clear gain (mean > SD)
   ⚠️ METHOD DISAGREEMENT (wins or sign differ) — both shown

PART A (global) − I (cal) [mission]
   seed 42   folds [1, 2, 3, 4]  mean Δ -0.0204  SD 0.0587  wins 1/4  p 0.537  

## 8. Learned weights and the capacity table

In [15]:
# ==============================================================================
# 8. LEARNED WEIGHTS + CAPACITY
# ==============================================================================
print("PART A global weights (A, I, F1) per evaluated pair:")
print(pd.DataFrame([{"fold": k, "seed": s, **{f"w_{e}": FIT[(k, s)]["wA"][i] for i, e in enumerate(EXPERTS)}}
                    for k, s in EVAL_PAIRS]).round(3).to_string(index=False))
print("\nPART B class-aware weights W[expert, class] per evaluated pair (each column sums to 1):")
for k, s in EVAL_PAIRS:
    W = FIT[(k, s)]["WB"]
    assert np.allclose(W.sum(0), 1)
    print(f"  fold {k} seed {s}: " + " | ".join(f"c{c}: " + "/".join(f"{W[i, c]:.2f}" for i in range(len(EXPERTS)))
                                               for c in range(N_CLASSES)) + "   (A/I/F1)")
_WB = np.stack([FIT[p]["WB"] for p in EVAL_PAIRS])
print("\nPART B across the evaluated pairs, mean ± SD of W[expert, class]:")
print(pd.DataFrame([[f"{_WB[:, i, c].mean():.2f} ± {_WB[:, i, c].std():.2f}" for c in range(N_CLASSES)] for i in range(len(EXPERTS))],
                   index=EXPERTS, columns=[f"class {c}" for c in range(N_CLASSES)]).to_string())
_dev = np.mean([np.abs(FIT[p]["WB"] - FIT[p]["wA"][:, None]).sum(0).mean() for p in EVAL_PAIRS])
print(f"Mean L1 distance between a Part B class column and the same pair's Part A weights: {_dev:.3f} "
      "(0 = Part B collapsed onto Part A)")

# ---- capacity table: experts from the saved CSVs (NOT instantiated); v32 router from its architecture ----
ROUTER_PARAMS = 27 * 32 + 32 + 32 * 3 + 3           # v32 router: Linear(27->32) + Linear(32->3) = 995
_pc = pd.concat([pd.read_csv(V29_RUNS_CSV).groupby("model").n_params.first(),
                 pd.read_csv(V30_RUNS_CSV).groupby("model").n_params.first() if V30_RUNS_CSV.exists() else pd.Series(dtype=float)])
_pc = _pc[~_pc.index.duplicated()]
CAPACITY = [(name, int(n), "expert / fusion model (saved run CSV)") for name, n in _pc.items()] + [
    ("ROUTER (v32)", ROUTER_PARAMS, "per-window MLP 27->32->3; window-dependent weights"),
    ("EQUAL", 0, "fixed 1/3 each"),
    ("TUNED", 3, f"3 simplex weights (2 free), grid of {len(SIMPLEX)}"),
    ("PART A (global)", len(EXPERTS), "3 logits -> softmax (2 effective)"),
    ("PART B (class-aware)", len(EXPERTS) * N_CLASSES, "3 x 5 logits -> per-class softmax (10 effective)")]
CAP_TAB = pd.DataFrame(CAPACITY, columns=["model / combiner", "parameters", "note"])
print("\nCapacity (combiners also share 3 calibration temperatures per fold/seed, fit on CAL_HALF):")
print(CAP_TAB.to_string(index=False, formatters={"parameters": "{:,}".format}))

PART A global weights (A, I, F1) per evaluated pair:
 fold  seed   w_A   w_I  w_F1
    0   123 0.000 0.729 0.271
    1    42 0.000 0.572 0.428
    2    42 0.000 0.642 0.358
    2   123 0.061 0.719 0.220
    3    42 0.000 0.770 0.230
    3   123 0.000 0.966 0.034
    4    42 0.000 0.482 0.518
    4   123 0.000 0.567 0.433

PART B class-aware weights W[expert, class] per evaluated pair (each column sums to 1):
  fold 0 seed 123: c0: 0.00/0.00/1.00 | c1: 0.00/0.86/0.13 | c2: 0.00/0.80/0.20 | c3: 0.00/0.31/0.69 | c4: 0.00/1.00/0.00   (A/I/F1)
  fold 1 seed 42: c0: 0.00/0.00/1.00 | c1: 0.02/0.98/0.01 | c2: 0.00/0.67/0.33 | c3: 0.00/0.00/1.00 | c4: 0.00/1.00/0.00   (A/I/F1)
  fold 2 seed 42: c0: 0.00/0.00/1.00 | c1: 0.00/0.77/0.23 | c2: 0.00/1.00/0.00 | c3: 0.00/0.14/0.86 | c4: 0.00/0.59/0.41   (A/I/F1)
  fold 2 seed 123: c0: 0.00/0.00/1.00 | c1: 0.00/1.00/0.00 | c2: 0.10/0.81/0.09 | c3: 0.14/0.01/0.85 | c4: 0.00/0.00/1.00   (A/I/F1)
  fold 3 seed 42: c0: 0.00/0.00/1.00 | c1: 0.00/0.39/0.61 

## 9. Summary and verdict

In [16]:
# ==============================================================================
# 9. SUMMARY + HOOKS
# ==============================================================================
def rrow(method, level):
    return RESULTS[(RESULTS.method == method) & (RESULTS.level == level)].iloc[0]


def prow(name, level="window"):
    return next(x for x in PAIRED if x["comparison"] == name and x["level"] == level)


def comp_line(name, level="window"):
    r = prow(name, level)
    return (f"{name} [{level}]: " + " | ".join(
        f"{meth} Δ {r[meth]['mean']:+.4f} SD {r[meth]['sd']:.4f} (wins {r[meth]['wins']}/{r[meth]['n']}, p {r[meth]['p']:.3f}) -> "
        f"{verdict(r[meth])}; {noise_rule(r[meth])}" for meth in ("pooled-probability", "seed-averaged-F1")) + f"  [{r['flag']}]"
        + "  | per seed: " + "; ".join(f"{s}: Δ {r[f'seed {s}']['mean']:+.4f} SD {r[f'seed {s}']['sd']:.4f} "
                                        f"({r[f'seed {s}']['wins']}/{r[f'seed {s}']['n']})" for s in SEEDS_EVAL))


def clear_gain(name, level="window"):
    r = prow(name, level)
    return all(r[m]["mean"] > 0 and np.isfinite(r[m]["sd"]) and r[m]["mean"] > r[m]["sd"] for m in ("pooled-probability", "seed-averaged-F1"))


def beats(name, level="window"):
    r = prow(name, level)
    return all(verdict(r[m]) == "BEATS" for m in ("pooled-probability", "seed-averaged-F1"))


_ba, _ai, _bi, _at = f"{PART_B} − {PART_A}", f"{PART_A} − I (cal)", f"{PART_B} − I (cal)", f"{PART_A} − TUNED"
_w = {m: rrow(m, "window") for m in METHODS}
_rba = prow(_ba)
_a_repro = all(verdict(prow(_at)[m]) == "no reliable difference" for m in ("pooled-probability", "seed-averaged-F1"))
VERDICT = (
    f"On this reduced set ({DENOM}), class-aware weighting (Part B, 15 parameters) "
    + ("CLEARLY BEATS" if clear_gain(_ba) else "does NOT clearly beat")
    + f" flat global weighting (Part A, 3 parameters): window macro-F1 {_w[PART_B].macro_f1:.3f} vs {_w[PART_A].macro_f1:.3f}; "
      f"paired Δ {_rba['pooled-probability']['mean']:+.4f} (SD {_rba['pooled-probability']['sd']:.4f}, pooled-probability) / "
      f"{_rba['seed-averaged-F1']['mean']:+.4f} (SD {_rba['seed-averaged-F1']['sd']:.4f}, seed-averaged-F1). "
    + ("" if clear_gain(_ba) else
       "This is CONSISTENT with the project's established pattern: more flexible combination mechanisms (the per-window "
       "router, cross-attention, independent gates) have repeatedly failed to beat simple fixed weights on this dataset, and "
       "per-class granularity is not rewarded either — it is not an unexpected failure. ")
    + f"Against I alone ({_w['I (cal)'].macro_f1:.3f}): Part A "
    + ("BEATS" if beats(_ai) else "clearly exceeds (mean Δ > SD) but does not meet the v32 BEATS rule" if clear_gain(_ai)
       else "does not reliably beat")
    + " it, and Part B " + ("BEATS" if beats(_bi) else "clearly exceeds (mean Δ > SD) but does not meet the v32 BEATS rule"
                              if clear_gain(_bi) else "does not reliably beat")
    + f" it. Part A {'reproduces' if _a_repro else 'does NOT cleanly reproduce'} TUNED on the test folds "
      f"(TUNED {_w['TUNED'].macro_f1:.3f} vs Part A {_w[PART_A].macro_f1:.3f}). "
      f"Caveats: every evaluated pair used a fitting pool with at most {int(POOL_TAB.pool_days.max())} of 4 other days; folds "
      "0 and 1 are represented by a single seed; with n = 4 folds per seed and n = 5 combined, the paired tests have little "
      "power; CAL/ROUTER halves come from other days' saved test windows (not a nested CV). These numbers are not comparable "
      "to v32's 10-pair results.")

L = ["WHAT v33 TELLS US", "",
     f"Files: {len(EXPECTED_FILES) - len(MISSING)}/{len(EXPECTED_FILES)} found in {NPZ_DIR}; missing: "
     + (", ".join(f"{m}_{k}_{s}.npz" for m, k, s in MISSING) or "none") + ". version 29/Npz never read.",
     f"Evaluated pairs ({N_EVAL}): {EVAL_PAIRS}; excluded from everything: {EXCLUDED_PAIRS}.",
     "Pools: " + "; ".join(f"{r.fold}/{r.seed}: {r.pool_days} days (excl. {r.days_excluded})" for r in POOL_TAB.itertuples()), "",
     "Sanity (Part A vs TUNED): " + SANITY_STATEMENT, "",
     f"Pooled window macro-F1 [95% CI] — {DENOM}:",
     "  " + "; ".join(f"{m} {_w[m].macro_f1:.3f} [{_w[m].f1_ci_low:.3f}, {_w[m].f1_ci_high:.3f}]" for m in METHODS),
     "Pooled mission macro-F1 (context): " + "; ".join(f"{m} {rrow(m, 'mission').macro_f1:.3f}" for m in METHODS), "",
     "Paired (window):"] + ["  " + comp_line(f"{a} − {b}") for a, b in COMPARISONS] + [
     "", "Capacity: " + "; ".join(f"{r[0]} {r[1]:,}" for r in CAPACITY),
     "CAPACITY DISCIPLINE: Part B is a deliberate small step (3 -> 15 parameters) up from Part A; it is credited only if "
     "mean Δ over Part A exceeds its own SD across folds under both combined methods.", "",
     "VERDICT: " + VERDICT, "",
     f"Total notebook time: {(time.time() - T0_NOTEBOOK) / 60:.1f} min."]
SUMMARY = "\n".join(L)
print(SUMMARY)
(WORK / "v33_summary.txt").write_text(SUMMARY, encoding="utf-8")

# ---- provenance hook: every .npz read came from version 32/v32_npz, none from version 29/Npz ----
hook("no_v29_npz", len(LOADED_NPZ) > 0 and all(p.parent == NPZ_DIR and FORBIDDEN_NPZ_REL not in p.as_posix() for p in LOADED_NPZ),
     f"{len(LOADED_NPZ)} .npz reads, all from {NPZ_DIR}; none from {FORBIDDEN_NPZ_REL}")

print("\n" + "=" * 90 + "\nVERIFICATION HOOKS\n" + "=" * 90)
for name, res in HOOKS.items():
    print(f"  {name:<16} {'PASS' if all(ok for ok, _ in res) else 'FAIL'}  ({len(res)} checks)")
    for ok, msg in res:
        if not ok:
            print(f"      FAIL: {msg}")
print("\nWritten to", WORK, ":", ", ".join(p.name for p in sorted(WORK.glob("v33_*"))))

WHAT v33 TELLS US

Files: 28/30 found in /kaggle/working/AeroSync-Mamba/version 32/v32_npz; missing: I_0_42.npz, I_1_123.npz. version 29/Npz never read.
Evaluated pairs (8): [(0, 123), (1, 42), (2, 42), (2, 123), (3, 42), (3, 123), (4, 42), (4, 123)]; excluded from everything: [(0, 42), (1, 123)].
Pools: 0/123: 3 days (excl. 2021-12-31); 1/42: 3 days (excl. 2021-12-22); 2/42: 3 days (excl. 2021-12-22); 2/123: 3 days (excl. 2021-12-31); 3/42: 3 days (excl. 2021-12-22); 3/123: 3 days (excl. 2021-12-31); 4/42: 3 days (excl. 2021-12-22); 4/123: 3 days (excl. 2021-12-31)

Sanity (Part A vs TUNED): Part A vs TUNED re-fit on the same reduced pool: median L1 0.039 (max 0.060); Part A's ROUTER_HALF NLL minus TUNED's: max -0.00001 (no implementation failures). Part A vs v32's printed TUNED (4-day pool): median L1 0.114 (max 0.367); v33's own TUNED refit vs v32's: median L1 0.100.

Pooled window macro-F1 [95% CI] — pooled over 8 of 10 fold/seed pairs (NOT 10); fold 0 windows by seed [123]; fold 1